In [ ]:
print ("hello")

In [ ]:
%pip install matplotlib

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
x=[1,2,3,4,5]
y=[2,4,6,8,10]
plt.plot(x,y)
plt.show()

In [ ]:
def f(x):
    return 3*x**2 -4*x +5

In [ ]:
f(3.0)

In [ ]:
xs=np.arange(-5,5,0.25)
ys=f(xs)
ys

In [ ]:
%pip list

In [ ]:
plt.plot(xs,ys)

In [ ]:
h=0.000001
x=2/3
(f(x+h)-f(x))/h

In [ ]:
#more complex

a=2.0
b=-3.0
c=10.0
d=a*b+c
print(d)

In [ ]:
h=0.0001

a=2.0
b=-3.0
c=10.0

d1=a*b+c
c+=h
d2=a*b+c

print(d1,d2,(d2-d1)/h)



In [ ]:
import math
class Value:

    def __init__(self,data,_children=(),_op='',label=''):
        self.data=data
        self._prev=set(_children)
        self._backward = lambda: None
        self.grad=0
        self._op=_op
        self.label=label

    def __repr__(self):
        return f"Value(data={self.data})"
    
    def __add__(self,other):
        out=Value(self.data+other.data,(self,other),'+')
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out
    
    def __mul__(self,other):
        out=Value(self.data*other.data,(self,other),'*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out
    
    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
        out = Value(t, (self, ), 'tanh')

        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out
    
    def backward(self):
    
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

a=Value(2.0,label='a')
b=Value(-3.0,label='b')
c=Value(10.0,label='c')
a.__add__(b)

e=a*b;e.label='e'
d=e+c;d.label='d'

f=Value(-2.0,label='f')
L=f*d;L.label='L'


In [ ]:
d._prev

In [ ]:
d._op

In [ ]:
from graphviz import Digraph

def trace(root):
    #builds set of all nodes and edges in a graph
    nodes,edges=set(),set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child,v))
                build(child)
    build(root)
    return nodes,edges

def draw_dot(root):
    dot =Digraph(format='svg',graph_attr={'rankdir':'LR'})

    nodes,edges=trace(root)
    for n in nodes:
        uid=str(id(n))
        dot.node(name=uid,label="{%s|data %.4f|grad %.4f}"%(n.label,n.data,n.grad ),shape='record')
        if n._op:
            dot.node(name=uid+n._op,label=n._op)
            dot.edge(uid+n._op,uid)

    for n1,n2 in edges:
        dot.edge(str(id(n1)),str(id(n2))+n2._op)

    return dot

In [ ]:
L.grad=1.0
d.grad=-2.0
f.grad=4.0
e.grad=d.grad
c.grad=d.grad
a.grad=e.grad*b.data
b.grad=e.grad*a.data

In [ ]:
draw_dot(L)

In [ ]:
a.data+=0.01*a.grad
b.data+=0.01*b.grad
c.data+=0.01*c.grad
f.data+=0.01*f.grad

e=a*b
d=e+c
L=f*d

draw_dot(L)

In [ ]:
# def lol1():
#     L=D*f

#     dL/dd= ? f

#     (f(x+h)-f(x))/h
#     ((d+h)*f - d*f)/h
#     (d*f + h*f - d*f)/h
#     (h*f)/h
#     f
    
def lol():

    h=0.001
    a=Value(2.0,label='a')
    b=Value(-3.0,label='b')
    c=Value(10.0,label='c')
    a.__add__(b)

    e=a*b;e.label='e'
    d=e+c;d.label='d'

    f=Value(-2.0,label='f')
    L=f*d;L.label='L'
    L1=L.data

#----------------------------

    a=Value(2.0,label='a')
    b=Value(-3.0,label='b')
    c=Value(10.0,label='c')
    a.__add__(b)

    e=a*b;e.label='e'
    d=e+c;d.label='d'
    d.data+=h

    f=Value(-2.0,label='f')
    L=f*d;L.label='L'
    L2=L.data

    print((L2-L1)/h)

lol()


In [ ]:
plt.plot(np.arange(-5,5,0.2), np.tanh(np.arange(-5,5,0.2))); plt.grid()

In [ ]:
# inputs x1,x2
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')
# weights w1,w2
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')
# bias of the neuron
b = Value(6.8813735870195432, label='b')
# x1*w1 + x2*w2 + b
x1w1 = x1*w1; x1w1.label = 'x1*w1'
x2w2 = x2*w2; x2w2.label = 'x2*w2'
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1*w1 + x2*w2'
n = x1w1x2w2 + b; n.label = 'n'
o = n.tanh(); o.label = 'o'

In [ ]:

x1w1.grad = 0.5
x2w2.grad = 0.5
x1w1x2w2.grad = 0.5
b.grad = 0.5
n.grad = 0.5
o.grad = 1.0
x1.grad = w1.data * x1w1.grad
w1.grad = x1.data * x1w1.grad
x2.grad = w2.data * x2w2.grad
w2.grad = x2.data * x2w2.grad

In [ ]:
o.backward()

In [ ]:
draw_dot(o)

In [ ]:
a = Value(3.0, label='a')
b = a + a   ; b.label = 'b'
b.backward()
draw_dot(b)

In [ ]:
a = Value(-2.0, label='a')
b = Value(3.0, label='b')
d = a * b    ; d.label = 'd'
e = a + b    ; e.label = 'e'
f = d * e    ; f.label = 'f'

f.backward()

draw_dot(f)

In [ ]:
import torch

In [ ]:
x1 = torch.Tensor([2.0]).double()                ; x1.requires_grad = True
x2 = torch.Tensor([0.0]).double()                ; x2.requires_grad = True
w1 = torch.Tensor([-3.0]).double()               ; w1.requires_grad = True
w2 = torch.Tensor([1.0]).double()                ; w2.requires_grad = True
b = torch.Tensor([6.8813735870195432]).double()  ; b.requires_grad = True
n = x1*w1 + x2*w2 + b
o = torch.tanh(n)

print(o.data.item())
o.backward()

print('---')
print('x2', x2.grad.item())
print('w2', w2.grad.item())
print('x1', x1.grad.item())
print('w1', w1.grad.item())

In [ ]:
temp=torch.Tensor([2,3,4,5])
temp.view(1,2,2)